In [1]:
from pathlib import Path
from datetime import datetime
import gc
import shutil
import warnings
import inspect

import numpy as np
import pandas as pd
import torch

import model_base as mb

warnings.simplefilter("default")


# ============================================================
# Run switches
# ============================================================
# Final default below is prepared for running S1 now.
# If you want only dry-run again, set RUN_S0_DRYRUN=True and RUN_S1_MAIN_CV=False.

RUN_S0_DRYRUN = False

# Keep this False so S1 is controlled explicitly.
AUTO_RUN_S1_AFTER_S0_PASS = False

RUN_S1_MAIN_CV = False
RUN_S2_BASELINES = False
RUN_S3_ARCH_ABLATION = True
RUN_S3B_TEMPORAL_SENSITIVITY = False
RUN_S4_MODAL_ABLATION = False
RUN_S5_LANDMARK_QUALITY = False
RUN_S6_CROSS_DATASET = False
RUN_S7_MULTI_SEED = False
RUN_S8_COMPUTE_COST = False
RUN_S9_CALIBRATION_EXPORT = False
RUN_S10_INTERPRETABILITY = False
RUN_S11_EXPORT_PAPER_TABLES = False

CONFIRM_LONG_RUNS = True
RUN_S1_PREFLIGHT_CHECK = False
RUN_PRE_EXPERIMENT_CAPACITY_CHECK = False


# ============================================================
# Global setup
# ============================================================

SEED = 42
mb.set_all_seeds(SEED)

DEVICE = mb.get_device()

RUN_DATE_OVERRIDE = "20260501"

if RUN_DATE_OVERRIDE is None:
    RUN_DATE = datetime.now().strftime("%Y%m%d")
else:
    RUN_DATE = str(RUN_DATE_OVERRIDE)

OUTPUT_ROOT = Path(f"outputs/revision_final_{RUN_DATE}")
PROCESSED_DIR = Path("data/processed")

CONFIG = {
    "sequence_length": 50,
    "stride": 25,
    "hidden_features": 128,
    "num_heads": 4,
    "dropout_i3d": 0.4,
    "dropout_rlt": 0.3,
    "batch_size": 16,
    "epochs": 100,
    "patience": 15,
    "lr": 1e-4,
    "weight_decay": 1e-2,
    "validation_split": 0.28,
    "threshold_strategy": "validation_youden",
    "early_stopping_metric": "validation_auc",
    "main_seed": 42,
    "multiseeds": [42, 123, 2024, 2025, 3407],
    "fallback_multiseeds": [42, 123, 2024],
    "i3d_cv": {"n_splits": 5, "n_repeats": 1},
    "rlt_cv": {"n_splits": 5, "n_repeats": 3},
    "rlt_ablation_cv": {"n_splits": 5, "n_repeats": 1},
    "full_model_use_cross_modal": False,
    "transformer_num_layers": 4,
    "s7_multiseed_split_protocol": (
        "All seeds share identical video-level splits generated with seed=42 "
        "to isolate random initialization/training stochasticity from data partition variance."
    ),
    "s6_cross_dataset_protocol": (
        "MSTGNet source-only is trained on source train and thresholded on source validation; "
        "traditional temporal-stat source-only and diagonal CORAL are run separately. "
        "Target test is never used for training, scaling, thresholding, or adaptation."
    ),
    "notes": {
        "canonical_split_files": (
            "Use *_splits_main_seed42.json or *_splits_CANONICAL_main_seed42.json "
            "as canonical split definitions. Files named *_splits_seed*.json may be "
            "created internally by run_cv_experiment and can contain only remaining splits."
        ),
        "s6_mstgnet_source_only": (
            "S6 MSTGNet source-only is trained specifically on the source dataset for transfer. "
            "It does not reuse S1 best checkpoints."
        ),
        "manuscript_consistency": (
            "Manuscript must say sigmoid binary logit, not softmax; traditional ML uses "
            "temporal-stat 13,527-D descriptors, not mean-only 1,503-D; mixed precision "
            "should not be claimed unless AMP is added; dropout is dataset-specific "
            "I3D=0.4 and RLT=0.3."
        ),
    },
}

DATASETS = ["I3D", "RLT"]

for subdir in [
    "config",
    "config/splits",
    "tables",
    "predictions",
    "fold_results",
    "stats",
    "checkpoints",
    "interpretability",
    "cross_dataset",
    "landmark_quality",
    "compute_cost",
    "figures_data",
    "logs",
    "manuscript_exports",
]:
    mb.ensure_dir(OUTPUT_ROOT / subdir)

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DEVICE:", DEVICE)

manifest = mb.create_experiment_manifest(
    output_dir=OUTPUT_ROOT,
    config=CONFIG,
    run_name="revision_final_v5",
)
mb.save_experiment_manifest(manifest, OUTPUT_ROOT)


# ============================================================
# model_base API validation
# ============================================================

required_mb_functions = [
    "dry_run_one_fold",
    "run_cv_experiment",
    "run_deep_learning_baselines",
    "run_traditional_ml_baseline",
    "run_cross_dataset_methods",
    "run_cross_dataset_mstgnet_source_only",
    "compute_landmark_quality_table",
    "compute_landmark_quality_prediction_correlation",
    "compute_gradient_importance_from_checkpoint",
    "save_gradient_importance",
    "select_checkpoint_validation_only",
    "compare_models_statistically",
    "compute_repeat_level_ci",
    "save_repeat_level_ci_table",
    "audit_video_split_overlap",
    "get_video_labels_from_data",
    "filter_completed_splits",
    "summarize_cv_results",
    "default_traditional_ml_models",
    "make_video_level_splits",
    "save_splits",
    "load_splits",
    "prepare_sequences",
    "compute_model_capacity_table",
    "wilcoxon_signed_rank_test",
    "paired_ttest",
    "safe_json_dump",
    "standardize_model_names_for_paper",
    "build_mstgnet_factory",
    "build_transformer_factory",
    "build_lstm_factory",
    "build_bilstm_factory",
    "build_gru_factory",
    "build_cnn_lstm_factory",
]

for name in required_mb_functions:
    assert hasattr(mb, name), f"model_base missing: {name}"

print("PASS: model_base API compatible with notebook")


# ============================================================
# Helper functions
# ============================================================

def transformer_model_name() -> str:
    return f"Transformer_L{int(CONFIG['transformer_num_layers'])}"


def dataset_dropout(dataset_name: str) -> float:
    name = str(dataset_name).upper()
    if name == "I3D":
        return float(CONFIG["dropout_i3d"])
    if name == "RLT":
        return float(CONFIG["dropout_rlt"])
    return 0.3


def dataset_cv_params(dataset_name: str) -> dict:
    name = str(dataset_name).upper()
    if name == "I3D":
        return CONFIG["i3d_cv"]
    if name == "RLT":
        return CONFIG["rlt_cv"]
    raise ValueError(f"Unknown dataset: {dataset_name}")


def active_feature_dim(dataset_name: str) -> int:
    if "data_dict" in globals() and dataset_name in data_dict:
        return int(len(data_dict[dataset_name]["coord_cols_active"]))
    return 1503


def build_mstgnet_for_dataset(
    dataset_name: str,
    data: dict,
    *,
    use_graph: bool = True,
    use_temporal: bool = True,
    use_pooling: bool = True,
    use_cross_modal: bool = False,
    enabled_modalities=("face", "eyes", "body"),
    hidden_dim: int = None,
    dropout: float = None,
):
    if hidden_dim is None:
        hidden_dim = int(CONFIG["hidden_features"])
    if dropout is None:
        dropout = dataset_dropout(dataset_name)

    return mb.build_mstgnet_factory(
        input_dim=int(len(data["coord_cols_active"])),
        modality_slices=data["active_modality_slices"],
        hidden_dim=int(hidden_dim),
        num_heads=int(CONFIG["num_heads"]),
        dropout=float(dropout),
        enabled_modalities=tuple(enabled_modalities),
        use_graph=bool(use_graph),
        use_temporal=bool(use_temporal),
        use_pooling=bool(use_pooling),
        use_cross_modal=bool(use_cross_modal),
    )


def build_transformer_for_dataset(
    dataset_name: str,
    *,
    input_dim: int = None,
    hidden_dim: int = None,
    num_layers: int = None,
    dropout: float = None,
):
    if input_dim is None:
        input_dim = active_feature_dim(dataset_name)
    if hidden_dim is None:
        hidden_dim = int(CONFIG["hidden_features"])
    if dropout is None:
        dropout = dataset_dropout(dataset_name)
    if num_layers is None:
        num_layers = int(CONFIG["transformer_num_layers"])

    return mb.build_transformer_factory(
        input_dim=int(input_dim),
        hidden_dim=int(hidden_dim),
        num_heads=int(CONFIG["num_heads"]),
        num_layers=int(num_layers),
        dropout=float(dropout),
    )


def require_long_run(section_name: str):
    if not CONFIRM_LONG_RUNS:
        raise RuntimeError(
            f"{section_name} is a long run. Set CONFIRM_LONG_RUNS=True before running."
        )


def read_csv_if_exists(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


def copy_if_exists(src, dst):
    src = Path(src)
    dst = Path(dst)
    if src.exists():
        mb.ensure_dir(dst.parent)
        shutil.copy(src, dst)
        print(f"Copied: {src} -> {dst}")
    else:
        print(f"Skip copy, not found: {src}")


def save_table(df: pd.DataFrame, name: str) -> Path:
    path = OUTPUT_ROOT / "tables" / name
    mb.ensure_dir(path.parent)
    df.to_csv(path, index=False)
    print(f"Saved table: {path} shape={df.shape}")
    return path


def fold_results_file(model_name: str, dataset_name: str) -> Path:
    return OUTPUT_ROOT / "fold_results" / f"{model_name}_{dataset_name}.csv"


def combine_fold_results(files_or_patterns) -> pd.DataFrame:
    rows = []

    for item in files_or_patterns:
        item_path = Path(item)
        item_str = str(item_path)

        if "*" in item_str or "?" in item_str:
            parent = item_path.parent
            pattern = item_path.name
            matched = sorted(parent.glob(pattern))
        else:
            matched = [item_path]

        for p in matched:
            if p.exists():
                try:
                    rows.append(pd.read_csv(p))
                except Exception as e:
                    warnings.warn(f"Could not read {p}: {e}", RuntimeWarning)

    if rows:
        return pd.concat(rows, ignore_index=True)

    return pd.DataFrame()


def drop_duplicate_fold_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    key_cols = [
        "dataset",
        "source_dataset",
        "target_dataset",
        "model",
        "seed",
        "repeat",
        "fold",
    ]
    existing = [c for c in key_cols if c in df.columns]

    required_base = {"dataset", "model", "seed", "repeat", "fold"}
    if required_base.issubset(set(existing)):
        return df.drop_duplicates(subset=existing, keep="last").reset_index(drop=True)

    return df.drop_duplicates().reset_index(drop=True)


def drop_duplicate_prediction_rows(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()

    candidate_keys = [
        "dataset",
        "source_dataset",
        "target_dataset",
        "model",
        "seed",
        "repeat",
        "fold",
        "split",
        "video_id",
    ]
    existing = [c for c in candidate_keys if c in df.columns]

    required_base = {"dataset", "model", "seed", "repeat", "fold", "split", "video_id"}
    if required_base.issubset(set(existing)):
        return df.drop_duplicates(subset=existing, keep="last").reset_index(drop=True)

    return df.drop_duplicates().reset_index(drop=True)


def summarize_results(
    df: pd.DataFrame,
    *,
    output_name: str,
    metric_cols=None,
    group_cols=None,
) -> pd.DataFrame:
    if df is None or len(df) == 0:
        out = pd.DataFrame()
    else:
        out = mb.summarize_cv_results(
            df,
            metric_cols=metric_cols,
            group_cols=group_cols,
        )
    save_table(out, output_name)
    return out


def make_or_load_splits(
    dataset_name: str,
    data: dict,
    *,
    n_splits: int,
    n_repeats: int,
    seed: int = 42,
    suffix: str = "main",
):
    split_path = OUTPUT_ROOT / "config" / "splits" / f"{dataset_name}_splits_{suffix}_seed{seed}.json"

    if split_path.exists():
        print(f"Loading existing splits: {split_path}")
        return mb.load_splits(split_path)

    video_ids, y_video = mb.get_video_labels_from_data(data)

    splits = mb.make_video_level_splits(
        video_ids,
        y_video,
        n_splits=int(n_splits),
        n_repeats=int(n_repeats),
        val_ratio=float(CONFIG["validation_split"]),
        seed=int(seed),
    )
    mb.save_splits(splits, split_path)
    print(f"Saved splits: {split_path}")
    return splits


def first_repeat_only(splits):
    return [s for s in splits if int(s["repeat"]) == 1]


def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_torch_load_checkpoint_local(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def save_full_model_reference_for_ablation(dataset_name: str):
    src_full = OUTPUT_ROOT / "fold_results" / f"MSTGNet_{dataset_name}.csv"
    dst_full_ref = OUTPUT_ROOT / "fold_results" / f"full_model_reference_{dataset_name}.csv"

    if src_full.exists():
        df_full = pd.read_csv(src_full)

        if "repeat" in df_full.columns:
            df_full_ref = df_full[df_full["repeat"].astype(int) == 1].copy()
        else:
            df_full_ref = df_full.copy()

        df_full_ref.to_csv(dst_full_ref, index=False)
        print(
            f"Saved full model reference for ablation: {dst_full_ref} "
            f"shape={df_full_ref.shape}"
        )
    else:
        print(f"Skip full model reference, not found: {src_full}")


def require_full_model_reference(dataset_name: str):
    ref_path = OUTPUT_ROOT / "fold_results" / f"full_model_reference_{dataset_name}.csv"
    if not ref_path.exists():
        raise FileNotFoundError(
            f"Missing full-model reference for {dataset_name}: {ref_path}. "
            "Run S1 first, or create full_model_reference_{dataset}.csv from repeat=1."
        )
    return ref_path


def get_param_column(df: pd.DataFrame):
    for c in ["n_trainable_params", "n_params", "trainable_params"]:
        if c in df.columns:
            return c
    return None


def deep_learning_baseline_names() -> list:
    return [
        "LSTM",
        "BiLSTM",
        "GRU",
        "CNN-LSTM",
        transformer_model_name(),
    ]


def traditional_ml_model_names() -> list:
    return list(mb.default_traditional_ml_models(seed=SEED).keys())


def cross_dataset_model_names() -> list:
    return [
        "MSTGNet_SourceOnly",
        "LogReg_temporal_stat_SourceOnly",
        "LogReg_temporal_stat_CORAL",
        "RF_temporal_stat_SourceOnly",
        "RF_temporal_stat_CORAL",
        "SVM_temporal_stat_SourceOnly",
        "SVM_temporal_stat_CORAL",
        "GradBoost_temporal_stat_SourceOnly",
        "GradBoost_temporal_stat_CORAL",
    ]


def architecture_ablation_model_names() -> list:
    return [
        "MSTGNet_NoGraph",
        "MSTGNet_NoTemporal",
        "MSTGNet_NoPooling",
        "MSTGNet_CrossModalAblation",
    ]


def modality_ablation_model_names() -> list:
    return [
        "MSTGNet_FaceOnly",
        "MSTGNet_EyesOnly",
        "MSTGNet_BodyOnly",
        "MSTGNet_FaceEyes",
        "MSTGNet_FaceBody",
        "MSTGNet_EyesBody",
    ]


def all_baseline_model_names() -> list:
    return ["MSTGNet"] + deep_learning_baseline_names() + traditional_ml_model_names()


def baseline_fold_result_files() -> list:
    files = []
    for model_name in all_baseline_model_names():
        for ds in DATASETS:
            files.append(OUTPUT_ROOT / "fold_results" / f"{model_name}_{ds}.csv")
    return files


def run_cv_resume_safe(
    *,
    dataset_name: str,
    model_name: str,
    model_factory,
    X_seq,
    y_seq,
    video_seq_ids,
    splits,
    seed: int,
    save_checkpoint: bool = True,
) -> pd.DataFrame:
    existing_path = fold_results_file(model_name, dataset_name)
    splits_to_run = mb.filter_completed_splits(
        splits,
        fold_results_path=existing_path,
        seed=seed,
    )

    df_new = pd.DataFrame()

    if len(splits_to_run) > 0:
        df_new = mb.run_cv_experiment(
            dataset_name=dataset_name,
            model_name=model_name,
            model_factory=model_factory,
            X_seq=X_seq,
            y_seq=y_seq,
            video_seq_ids=video_seq_ids,
            splits=splits_to_run,
            device=DEVICE,
            output_dir=OUTPUT_ROOT,
            batch_size=int(CONFIG["batch_size"]),
            epochs=int(CONFIG["epochs"]),
            patience=int(CONFIG["patience"]),
            lr=float(CONFIG["lr"]),
            weight_decay=float(CONFIG["weight_decay"]),
            gradient_clip=5.0,
            seed=seed,
            save_checkpoint=save_checkpoint,
        )
    else:
        print(f"{model_name} | {dataset_name}: all splits already complete.")

    if existing_path.exists():
        return pd.read_csv(existing_path)

    return df_new


def find_best_checkpoint(
    dataset_name: str,
    model_name: str = "MSTGNet",
    *,
    seed_whitelist=None,
    repeat_whitelist=None,
):
    path = OUTPUT_ROOT / "fold_results" / f"{model_name}_{dataset_name}.csv"

    return mb.select_checkpoint_validation_only(
        path,
        seed_whitelist=seed_whitelist,
        repeat_whitelist=repeat_whitelist,
        checkpoint_col="checkpoint_path",
    )


def standardize_table_for_paper_if_exists(table_name: str):
    path = OUTPUT_ROOT / "tables" / table_name
    if not path.exists():
        return

    df = pd.read_csv(path)
    if "model" in df.columns:
        df = mb.standardize_model_names_for_paper(df)
        df.to_csv(path, index=False)
        print(f"Standardized paper model names in: {path}")


# ============================================================
# Load processed datasets
# ============================================================

data_dict = {}

for ds in DATASETS:
    data_dict[ds] = mb.load_processed_dataset(
        ds,
        processed_dir=PROCESSED_DIR,
        expected_observed_features=1533,
        expected_active_features=1503,
    )


# ============================================================
# Dataset summary
# ============================================================

dataset_summary_rows = []

for ds, data in data_dict.items():
    video_ids, y = mb.get_video_labels_from_data(data)

    dataset_summary_rows.append(
        {
            "dataset": ds,
            "n_videos": int(len(video_ids)),
            "n_class_0": int(np.sum(y == 0)),
            "n_class_1": int(np.sum(y == 1)),
            "n_observed_features": int(len(data["coord_cols_observed"])),
            "n_active_features": int(len(data["coord_cols_active"])),
            "face_slice": str(data["active_modality_slices"].get("face")),
            "eyes_slice": str(data["active_modality_slices"].get("eyes")),
            "body_slice": str(data["active_modality_slices"].get("body")),
        }
    )

dataset_summary_df = pd.DataFrame(dataset_summary_rows)
save_table(dataset_summary_df, "dataset_summary.csv")
print(dataset_summary_df)


# ============================================================
# Splits
# ============================================================

splits_main = {}
splits_ablation = {}

for ds, data in data_dict.items():
    cvp = dataset_cv_params(ds)
    splits_main[ds] = make_or_load_splits(
        ds,
        data,
        n_splits=cvp["n_splits"],
        n_repeats=cvp["n_repeats"],
        seed=SEED,
        suffix="main",
    )

    canonical_path = (
        OUTPUT_ROOT
        / "config"
        / "splits"
        / f"{ds}_splits_CANONICAL_main_seed{SEED}.json"
    )
    mb.save_splits(splits_main[ds], canonical_path)

    splits_ablation[ds] = first_repeat_only(splits_main[ds])

print(
    {k: len(v) for k, v in splits_main.items()},
    {k: len(v) for k, v in splits_ablation.items()},
)

for ds in DATASETS:
    audit = mb.audit_video_split_overlap(splits_main[ds])
    print(ds)
    print(audit["status"].value_counts(dropna=False))
    assert (audit["status"] == "PASS").all(), audit

print("PASS: all main splits have no video-level overlap")


# ============================================================
# Sequence cache
# ============================================================

sequence_cache = {}


def get_sequences(dataset_name: str, *, seq_len=None, stride=None, cache_key=None):
    if seq_len is None:
        seq_len = int(CONFIG["sequence_length"])
    if stride is None:
        stride = int(CONFIG["stride"])

    if cache_key is None:
        cache_key = f"{dataset_name}_L{seq_len}_S{stride}"

    if cache_key in sequence_cache:
        return sequence_cache[cache_key]

    X_seq, y_seq, video_seq_ids = mb.prepare_sequences(
        data_dict[dataset_name],
        seq_len=int(seq_len),
        stride=int(stride),
        use_active_features=True,
        pad_short_videos=True,
        pad_mode="edge",
        verbose=True,
    )

    sequence_cache[cache_key] = (X_seq, y_seq, video_seq_ids)
    return X_seq, y_seq, video_seq_ids


# ============================================================
# Capacity check
# ============================================================

def capacity_factories_for_dataset(ds: str) -> dict:
    n_features = active_feature_dim(ds)
    hidden_dim = int(CONFIG["hidden_features"])
    dropout = dataset_dropout(ds)

    factories = {
        "MSTGNet": build_mstgnet_for_dataset(
            ds,
            data_dict[ds],
            use_graph=True,
            use_temporal=True,
            use_pooling=True,
            use_cross_modal=False,
            enabled_modalities=("face", "eyes", "body"),
        ),
        "LSTM": mb.build_lstm_factory(
            input_dim=n_features,
            hidden_dim=hidden_dim,
            dropout=dropout,
            num_layers=1,
        ),
        "BiLSTM": mb.build_bilstm_factory(
            input_dim=n_features,
            hidden_dim=hidden_dim,
            dropout=dropout,
            num_layers=1,
        ),
        "GRU": mb.build_gru_factory(
            input_dim=n_features,
            hidden_dim=hidden_dim,
            dropout=dropout,
            num_layers=1,
        ),
        "CNN-LSTM": mb.build_cnn_lstm_factory(
            input_dim=n_features,
            hidden_dim=hidden_dim,
            dropout=dropout,
            conv_kernel_size=5,
            lstm_layers=1,
            use_batchnorm=True,
        ),
    }

    layers = sorted(set([4, 6, int(CONFIG["transformer_num_layers"])]))
    for layer in layers:
        factories[f"Transformer_L{layer}"] = build_transformer_for_dataset(
            ds,
            num_layers=layer,
        )

    return factories


if RUN_PRE_EXPERIMENT_CAPACITY_CHECK:
    capacity_rows = []

    for ds in DATASETS:
        cdf = mb.compute_model_capacity_table(capacity_factories_for_dataset(ds))
        cdf["dataset"] = ds

        param_col = get_param_column(cdf)
        if param_col is not None:
            mstg_params = cdf.loc[cdf["model"].eq("MSTGNet"), param_col]
            if len(mstg_params) == 1:
                ref = float(mstg_params.iloc[0])
                cdf["param_ratio_vs_MSTGNet_manual"] = cdf[param_col] / max(ref, 1.0)

        capacity_rows.append(cdf)

    capacity_check_df = (
        pd.concat(capacity_rows, ignore_index=True)
        if capacity_rows else pd.DataFrame()
    )

    capacity_path = OUTPUT_ROOT / "compute_cost" / "capacity_check_pre_S2.csv"
    capacity_check_df.to_csv(capacity_path, index=False)

    print(f"Saved pre-S2 capacity check: {capacity_path}")
    print(capacity_check_df)

    param_col = get_param_column(capacity_check_df)
    if param_col is not None:
        for ds in DATASETS:
            d = capacity_check_df[capacity_check_df["dataset"].eq(ds)].copy()
            if len(d) == 0:
                continue

            mstg = d.loc[d["model"].eq("MSTGNet"), param_col]
            tr_name = transformer_model_name()
            tr = d.loc[d["model"].eq(tr_name), param_col]

            if len(mstg) == 1 and len(tr) == 1:
                ratio = float(tr.iloc[0]) / max(float(mstg.iloc[0]), 1.0)
                if ratio < 0.5 or ratio > 2.0:
                    warnings.warn(
                        f"{ds}: {tr_name} capacity ratio vs MSTGNet is {ratio:.3f}. "
                        "Consider changing CONFIG['transformer_num_layers'] to 4 or 6.",
                        RuntimeWarning,
                    )


# ============================================================
# S1 preflight check
# ============================================================

if RUN_S1_PREFLIGHT_CHECK:
    print("\n" + "=" * 80)
    print("S1 PREFLIGHT CHECK")
    print("=" * 80)

    print("run_cv_experiment:", inspect.signature(mb.run_cv_experiment))
    print("build_mstgnet_factory:", inspect.signature(mb.build_mstgnet_factory))

    assert hasattr(mb, "dry_run_one_fold")
    assert hasattr(mb, "run_cv_experiment")
    assert hasattr(mb, "filter_completed_splits")
    assert hasattr(mb, "build_mstgnet_factory")

    for ds in DATASETS:
        assert ds in data_dict
        assert len(data_dict[ds]["coord_cols_active"]) == 1503
        assert "face" in data_dict[ds]["active_modality_slices"]
        assert "eyes" in data_dict[ds]["active_modality_slices"]
        assert "body" in data_dict[ds]["active_modality_slices"]

    print("PASS: S1 preflight checks OK")


# ============================================================
# S0 — Dry-run
# ============================================================

s0_outputs = {}

if RUN_S0_DRYRUN:
    for ds in DATASETS:
        print("\n" + "=" * 80)
        print(f"S0 Dry-run: {ds}")
        print("=" * 80)

        X_seq, y_seq, video_seq_ids = get_sequences(ds)

        model_factory = build_mstgnet_for_dataset(
            ds,
            data_dict[ds],
            use_graph=True,
            use_temporal=True,
            use_pooling=True,
            use_cross_modal=False,
            enabled_modalities=("face", "eyes", "body"),
        )

        row = mb.dry_run_one_fold(
            dataset_name=ds,
            model_name="MSTGNet",
            model_factory=model_factory,
            X_seq=X_seq,
            y_seq=y_seq,
            video_seq_ids=video_seq_ids,
            splits=splits_ablation[ds],
            device=DEVICE,
            output_dir=OUTPUT_ROOT / f"S0_dryrun_{ds}",
            batch_size=int(CONFIG["batch_size"]),
            epochs=2,
            patience=2,
            lr=float(CONFIG["lr"]),
            weight_decay=float(CONFIG["weight_decay"]),
            gradient_clip=5.0,
            seed=SEED,
        )

        dry_status = str(row.get("dry_run_status", "")).upper()
        gate_status = "PASS" if dry_status.startswith("PASS") else "FAIL"
        best_val_auc = row.get("best_val_auc", np.nan)

        out = {
            "gate_status": gate_status,
            "dry_run_status": dry_status,
            "fold_result": row,
            "gate_checks": {
                "dry_run_status_pass": bool(gate_status == "PASS"),
                "auc_gt_0_5_desirable_not_hard_gate": bool(
                    np.isfinite(best_val_auc) and float(best_val_auc) > 0.5
                ),
            },
        }

        s0_outputs[ds] = out

        print(f"{ds} gate_status:", out["gate_status"])
        print("dry_run_status:", out["dry_run_status"])
        print("gate_checks:", out["gate_checks"])

    print(s0_outputs)


if RUN_S0_DRYRUN and AUTO_RUN_S1_AFTER_S0_PASS:
    s0_pass = all(
        str(s0_outputs.get(ds, {}).get("gate_status", "")).upper() == "PASS"
        for ds in DATASETS
    )

    print("AUTO_RUN_S1_AFTER_S0_PASS:", AUTO_RUN_S1_AFTER_S0_PASS)
    print("S0 all PASS:", s0_pass)

    if s0_pass:
        RUN_S1_MAIN_CV = True
        CONFIRM_LONG_RUNS = True
        print("S0 passed for all datasets. Enabling RUN_S1_MAIN_CV=True.")
    else:
        RUN_S1_MAIN_CV = False
        print("S0 did not pass for all datasets. S1 will not run.")
elif AUTO_RUN_S1_AFTER_S0_PASS and not RUN_S0_DRYRUN:
    print(
        "AUTO_RUN_S1_AFTER_S0_PASS=True but RUN_S0_DRYRUN=False; "
        "auto S1 gate skipped. RUN_S1_MAIN_CV remains:",
        RUN_S1_MAIN_CV,
    )


# ============================================================
# S1 — Main CV: MSTGNet
# ============================================================

if RUN_S1_MAIN_CV:
    require_long_run("S1_MAIN_CV")

    s1_results = []

    for ds in DATASETS:
        print("\n" + "#" * 90)
        print(f"S1 Main CV — MSTGNet Final — {ds}")
        print("#" * 90)

        X_seq, y_seq, video_seq_ids = get_sequences(ds)

        model_factory = build_mstgnet_for_dataset(
            ds,
            data_dict[ds],
            use_graph=True,
            use_temporal=True,
            use_pooling=True,
            use_cross_modal=False,
            enabled_modalities=("face", "eyes", "body"),
        )

        df = run_cv_resume_safe(
            dataset_name=ds,
            model_name="MSTGNet",
            model_factory=model_factory,
            X_seq=X_seq,
            y_seq=y_seq,
            video_seq_ids=video_seq_ids,
            splits=splits_main[ds],
            seed=SEED,
            save_checkpoint=True,
        )

        s1_results.append(df)
        clean_memory()

        copy_if_exists(
            OUTPUT_ROOT / "fold_results" / f"MSTGNet_{ds}.csv",
            OUTPUT_ROOT / "fold_results" / f"main_{ds}.csv",
        )

        save_full_model_reference_for_ablation(ds)

        copy_if_exists(
            OUTPUT_ROOT / "predictions" / f"MSTGNet_{ds}_video_predictions.csv",
            OUTPUT_ROOT / "predictions" / f"main_{ds}_video_predictions.csv",
        )

    s1_all = combine_fold_results([
        OUTPUT_ROOT / "fold_results" / "MSTGNet_I3D.csv",
        OUTPUT_ROOT / "fold_results" / "MSTGNet_RLT.csv",
    ])
    s1_all = drop_duplicate_fold_rows(s1_all)

    summarize_results(s1_all, output_name="main_results_summary.csv")
    print(s1_all)


# ============================================================
# S2 — Baselines: DL baselines + Traditional ML
# ============================================================

if RUN_S2_BASELINES:
    require_long_run("S2_BASELINES")

    s2_results = []

    for ds in DATASETS:
        print("\n" + "#" * 90)
        print(f"S2 — Deep Learning Baselines — {ds}")
        print("#" * 90)

        X_seq, y_seq, video_seq_ids = get_sequences(ds)

        df_dl = mb.run_deep_learning_baselines(
            dataset_name=ds,
            X_seq=X_seq,
            y_seq=y_seq,
            video_seq_ids=video_seq_ids,
            splits=splits_main[ds],
            device=DEVICE,
            output_dir=OUTPUT_ROOT,
            input_dim=active_feature_dim(ds),
            hidden_dim=int(CONFIG["hidden_features"]),
            num_heads=int(CONFIG["num_heads"]),
            transformer_num_layers=int(CONFIG["transformer_num_layers"]),
            dropout=dataset_dropout(ds),
            cnn_lstm_use_batchnorm=True,
            batch_size=int(CONFIG["batch_size"]),
            epochs=int(CONFIG["epochs"]),
            patience=int(CONFIG["patience"]),
            lr=float(CONFIG["lr"]),
            weight_decay=float(CONFIG["weight_decay"]),
            gradient_clip=5.0,
            seed=SEED,
            save_checkpoint=True,
            model_whitelist=deep_learning_baseline_names(),
        )
        s2_results.append(df_dl)
        clean_memory()

        print("\n" + "#" * 90)
        print(f"S2 — Traditional ML temporal-stat — {ds}")
        print("#" * 90)

        df_ml = mb.run_traditional_ml_baseline(
            dataset_name=ds,
            data=data_dict[ds],
            splits=splits_main[ds],
            output_dir=OUTPUT_ROOT,
            seed=SEED,
            feature_mode="temporal_stat",
        )
        s2_results.append(df_ml)
        clean_memory()

    baseline_all = combine_fold_results(baseline_fold_result_files())
    baseline_all = drop_duplicate_fold_rows(baseline_all)

    summarize_results(
        baseline_all,
        output_name="table_baseline_comparison.csv",
    )

    baseline_all.to_csv(
        OUTPUT_ROOT / "tables" / "baseline_fold_results_all.csv",
        index=False,
    )
    print(baseline_all)


# ============================================================
# S3 — Architecture ablation
# ============================================================

if RUN_S3_ARCH_ABLATION:
    require_long_run("S3_ARCH_ABLATION")

    for ds in DATASETS:
        require_full_model_reference(ds)

    arch_variants = {
        "MSTGNet_NoGraph": {
            "use_graph": False,
            "use_temporal": True,
            "use_pooling": True,
            "use_cross_modal": False,
        },
        "MSTGNet_NoTemporal": {
            "use_graph": True,
            "use_temporal": False,
            "use_pooling": True,
            "use_cross_modal": False,
        },
        "MSTGNet_NoPooling": {
            "use_graph": True,
            "use_temporal": True,
            "use_pooling": False,
            "use_cross_modal": False,
        },
        "MSTGNet_CrossModalAblation": {
            "use_graph": True,
            "use_temporal": True,
            "use_pooling": True,
            "use_cross_modal": True,
        },
    }

    for ds in DATASETS:
        print("\n" + "#" * 90)
        print(f"S3 Architecture Ablation — {ds}")
        print("#" * 90)

        X_seq, y_seq, video_seq_ids = get_sequences(ds)
        ab_splits = splits_ablation[ds]

        for model_name, kwargs in arch_variants.items():
            print("\n" + "-" * 70)
            print(f"{model_name} — {ds}")
            print("-" * 70)

            factory = build_mstgnet_for_dataset(
                ds,
                data_dict[ds],
                use_graph=kwargs["use_graph"],
                use_temporal=kwargs["use_temporal"],
                use_pooling=kwargs["use_pooling"],
                use_cross_modal=kwargs["use_cross_modal"],
                enabled_modalities=("face", "eyes", "body"),
            )

            _ = run_cv_resume_safe(
                dataset_name=ds,
                model_name=model_name,
                model_factory=factory,
                X_seq=X_seq,
                y_seq=y_seq,
                video_seq_ids=video_seq_ids,
                splits=ab_splits,
                seed=SEED,
                save_checkpoint=True,
            )
            clean_memory()

    arch_files = [
        OUTPUT_ROOT / "fold_results" / "full_model_reference_I3D.csv",
        OUTPUT_ROOT / "fold_results" / "full_model_reference_RLT.csv",
    ]
    for model_name in architecture_ablation_model_names():
        for ds in DATASETS:
            arch_files.append(OUTPUT_ROOT / "fold_results" / f"{model_name}_{ds}.csv")

    arch_all = combine_fold_results(arch_files)
    arch_all = drop_duplicate_fold_rows(arch_all)

    summarize_results(
        arch_all,
        output_name="table_architecture_ablation.csv",
    )

    arch_all.to_csv(
        OUTPUT_ROOT / "fold_results" / "arch_ablation_all.csv",
        index=False,
    )
    print(arch_all)


# ============================================================
# S3B — Temporal sensitivity
# ============================================================

if RUN_S3B_TEMPORAL_SENSITIVITY:
    require_long_run("S3B_TEMPORAL_SENSITIVITY")
    require_full_model_reference("I3D")

    ds = "I3D"

    temporal_settings = [
        {"name": "MSTGNet_Window100", "seq_len": 100, "stride": 50},
    ]

    for setting in temporal_settings:
        print("\n" + "#" * 90)
        print(f"S3B Temporal Sensitivity — {setting['name']} — {ds}")
        print("#" * 90)

        X_seq, y_seq, video_seq_ids = get_sequences(
            ds,
            seq_len=setting["seq_len"],
            stride=setting["stride"],
            cache_key=f"{ds}_L{setting['seq_len']}_S{setting['stride']}",
        )

        factory = build_mstgnet_for_dataset(
            ds,
            data_dict[ds],
            use_graph=True,
            use_temporal=True,
            use_pooling=True,
            use_cross_modal=False,
            enabled_modalities=("face", "eyes", "body"),
        )

        _ = run_cv_resume_safe(
            dataset_name=ds,
            model_name=setting["name"],
            model_factory=factory,
            X_seq=X_seq,
            y_seq=y_seq,
            video_seq_ids=video_seq_ids,
            splits=splits_ablation[ds],
            seed=SEED,
            save_checkpoint=True,
        )
        clean_memory()

    temporal_all = combine_fold_results([
        OUTPUT_ROOT / "fold_results" / "full_model_reference_I3D.csv",
        OUTPUT_ROOT / "fold_results" / "MSTGNet_Window100_I3D.csv",
    ])
    temporal_all = drop_duplicate_fold_rows(temporal_all)

    summarize_results(
        temporal_all,
        output_name="table_temporal_sensitivity.csv",
    )

    temporal_all.to_csv(
        OUTPUT_ROOT / "fold_results" / "temporal_sensitivity_all.csv",
        index=False,
    )
    print(temporal_all)


# ============================================================
# S4 — Modality ablation
# ============================================================

if RUN_S4_MODAL_ABLATION:
    require_long_run("S4_MODAL_ABLATION")

    for ds in DATASETS:
        require_full_model_reference(ds)

    modality_variants = {
        "MSTGNet_FaceOnly": ("face",),
        "MSTGNet_EyesOnly": ("eyes",),
        "MSTGNet_BodyOnly": ("body",),
        "MSTGNet_FaceEyes": ("face", "eyes"),
        "MSTGNet_FaceBody": ("face", "body"),
        "MSTGNet_EyesBody": ("eyes", "body"),
    }

    for ds in DATASETS:
        print("\n" + "#" * 90)
        print(f"S4 Modality Ablation — {ds}")
        print("#" * 90)

        X_seq, y_seq, video_seq_ids = get_sequences(ds)
        ab_splits = splits_ablation[ds]

        for model_name, modalities in modality_variants.items():
            print("\n" + "-" * 70)
            print(f"{model_name} modalities={modalities} — {ds}")
            print("-" * 70)

            factory = build_mstgnet_for_dataset(
                ds,
                data_dict[ds],
                use_graph=True,
                use_temporal=True,
                use_pooling=True,
                use_cross_modal=False,
                enabled_modalities=modalities,
            )

            _ = run_cv_resume_safe(
                dataset_name=ds,
                model_name=model_name,
                model_factory=factory,
                X_seq=X_seq,
                y_seq=y_seq,
                video_seq_ids=video_seq_ids,
                splits=ab_splits,
                seed=SEED,
                save_checkpoint=True,
            )
            clean_memory()

    modal_files = [
        OUTPUT_ROOT / "fold_results" / "full_model_reference_I3D.csv",
        OUTPUT_ROOT / "fold_results" / "full_model_reference_RLT.csv",
    ]
    for model_name in modality_ablation_model_names():
        for ds in DATASETS:
            modal_files.append(OUTPUT_ROOT / "fold_results" / f"{model_name}_{ds}.csv")

    modal_all = combine_fold_results(modal_files)
    modal_all = drop_duplicate_fold_rows(modal_all)

    summarize_results(
        modal_all,
        output_name="table_modality_ablation.csv",
    )

    modal_all.to_csv(
        OUTPUT_ROOT / "fold_results" / "modal_ablation_all.csv",
        index=False,
    )
    print(modal_all)


# ============================================================
# S5 — Landmark quality
# ============================================================

if RUN_S5_LANDMARK_QUALITY:
    quality_dfs = []

    for ds in DATASETS:
        print("\n" + "#" * 90)
        print(f"S5 Landmark Quality — {ds}")
        print("#" * 90)

        qdf = mb.compute_landmark_quality_table(
            data_dict[ds],
            dataset_name=ds,
            use_active_features=True,
            long_format=True,
        )
        quality_dfs.append(qdf)

    quality_all = (
        pd.concat(quality_dfs, ignore_index=True)
        if quality_dfs else pd.DataFrame()
    )

    quality_path = OUTPUT_ROOT / "landmark_quality" / "quality_by_video.csv"
    quality_all.to_csv(quality_path, index=False)
    print(f"Saved landmark quality table: {quality_path} shape={quality_all.shape}")

    if len(quality_all) > 0:
        quality_summary = (
            quality_all
            .groupby(["dataset", "modality"], dropna=False)
            .agg(
                n_videos=("video_id", "nunique"),
                finite_rate_mean=("finite_rate", "mean"),
                finite_rate_std=("finite_rate", "std"),
                nonfinite_rate_mean=("nonfinite_rate", "mean"),
                nonfinite_rate_std=("nonfinite_rate", "std"),
                zero_rate_mean=("zero_rate", "mean"),
                zero_rate_std=("zero_rate", "std"),
                mean_abs_value_mean=("mean_abs_value", "mean"),
                mean_abs_value_std=("mean_abs_value", "std"),
                std_abs_value_mean=("std_abs_value", "mean"),
                std_abs_value_std=("std_abs_value", "std"),
                mean_abs_velocity_mean=("mean_abs_velocity", "mean"),
                mean_abs_velocity_std=("mean_abs_velocity", "std"),
                std_abs_velocity_mean=("std_abs_velocity", "mean"),
                std_abs_velocity_std=("std_abs_velocity", "std"),
                p95_abs_velocity_mean=("p95_abs_velocity", "mean"),
                p95_abs_velocity_std=("p95_abs_velocity", "std"),
            )
            .reset_index()
        )
    else:
        quality_summary = pd.DataFrame()

    save_table(quality_summary, "table_landmark_quality.csv")
    print(quality_summary)

    quality_corr_dfs = []

    for ds in DATASETS:
        pred_path = OUTPUT_ROOT / "predictions" / f"MSTGNet_{ds}_video_predictions.csv"

        if not pred_path.exists():
            print(f"Skip quality-prediction correlation for {ds}; missing: {pred_path}")
            continue

        if len(quality_all) == 0:
            print(f"Skip quality-prediction correlation for {ds}; quality_all empty.")
            continue

        pred_df = pd.read_csv(pred_path)
        q_ds = quality_all[quality_all["dataset"].astype(str) == str(ds)].copy()

        if len(q_ds) == 0:
            print(f"Skip quality-prediction correlation for {ds}; no quality rows.")
            continue

        corr_df = mb.compute_landmark_quality_prediction_correlation(
            q_ds,
            pred_df,
            prediction_split="test",
            model_filter="MSTGNet",
            aggregate_predictions_per_video=True,
        )

        corr_df["dataset_requested"] = ds
        quality_corr_dfs.append(corr_df)

        corr_path = (
            OUTPUT_ROOT
            / "landmark_quality"
            / f"quality_prediction_correlation_{ds}.csv"
        )
        corr_df.to_csv(corr_path, index=False)
        print(f"Quality-prediction correlation {ds}: {corr_path} shape={corr_df.shape}")

    quality_corr_all = (
        pd.concat(quality_corr_dfs, ignore_index=True)
        if quality_corr_dfs else pd.DataFrame()
    )

    quality_corr_all.to_csv(
        OUTPUT_ROOT / "landmark_quality" / "quality_prediction_correlation_all.csv",
        index=False,
    )

    save_table(
        quality_corr_all,
        "table_landmark_quality_prediction_correlation.csv",
    )


# ============================================================
# S6 — Cross-dataset generalization
# ============================================================

if RUN_S6_CROSS_DATASET:
    require_long_run("S6_CROSS_DATASET")

    s6_pairs = [
        ("I3D", "RLT"),
        ("RLT", "I3D"),
    ]

    s6_results = []

    for source_ds, target_ds in s6_pairs:
        print("\n" + "#" * 90)
        print(f"S6 Cross-Dataset: {source_ds} -> {target_ds}")
        print("#" * 90)

        print("\n" + "-" * 70)
        print(f"S6A MSTGNet source-only: {source_ds} -> {target_ds}")
        print("-" * 70)

        source_model_factory = build_mstgnet_for_dataset(
            source_ds,
            data_dict[source_ds],
            use_graph=True,
            use_temporal=True,
            use_pooling=True,
            use_cross_modal=False,
            enabled_modalities=("face", "eyes", "body"),
        )

        df_mstg_source_only = mb.run_cross_dataset_mstgnet_source_only(
            source_dataset_name=source_ds,
            target_dataset_name=target_ds,
            source_data=data_dict[source_ds],
            target_data=data_dict[target_ds],
            target_splits=splits_ablation[target_ds],
            model_factory=source_model_factory,
            device=DEVICE,
            output_dir=OUTPUT_ROOT,
            seed=SEED,
            source_val_ratio=0.2,
            seq_len=int(CONFIG["sequence_length"]),
            stride=int(CONFIG["stride"]),
            batch_size=int(CONFIG["batch_size"]),
            epochs=int(CONFIG["epochs"]),
            patience=int(CONFIG["patience"]),
            lr=float(CONFIG["lr"]),
            weight_decay=float(CONFIG["weight_decay"]),
            gradient_clip=5.0,
            num_workers=0,
            model_name="MSTGNet_SourceOnly",
            save_checkpoint=True,
        )

        s6_results.append(df_mstg_source_only)
        clean_memory()

        print("\n" + "-" * 70)
        print(f"S6B Traditional ML source-only/CORAL: {source_ds} -> {target_ds}")
        print("-" * 70)

        df_trad_cd = mb.run_cross_dataset_methods(
            source_dataset_name=source_ds,
            target_dataset_name=target_ds,
            source_data=data_dict[source_ds],
            target_data=data_dict[target_ds],
            target_splits=splits_ablation[target_ds],
            output_dir=OUTPUT_ROOT,
            seed=SEED,
            source_val_ratio=0.2,
            models=mb.default_traditional_ml_models(seed=SEED),
            run_coral=True,
            coral_mode="diagonal",
            feature_mode="temporal_stat",
        )

        s6_results.append(df_trad_cd)
        clean_memory()

    s6_all = combine_fold_results([
        OUTPUT_ROOT / "fold_results" / "*_to_*.csv",
    ])
    s6_all = drop_duplicate_fold_rows(s6_all)

    save_table(s6_all, "table_cross_dataset.csv")
    s6_all.to_csv(
        OUTPUT_ROOT / "cross_dataset" / "cross_dataset_all_fold_results.csv",
        index=False,
    )

    print(s6_all)


# ============================================================
# S7 — Multi-seed robustness
# ============================================================

if RUN_S7_MULTI_SEED:
    require_long_run("S7_MULTI_SEED")

    tr_name = transformer_model_name()
    extra_seeds = [s for s in CONFIG["multiseeds"] if int(s) != 42]

    for seed in extra_seeds:
        seed = int(seed)

        print("\n" + "#" * 90)
        print(f"S7 Multi-seed additional seed={seed}")
        print("#" * 90)

        for ds in DATASETS:
            print("\n" + "=" * 80)
            print(f"S7 — {ds} — seed={seed}")
            print("=" * 80)

            X_seq, y_seq, video_seq_ids = get_sequences(ds)

            mstg_factory = build_mstgnet_for_dataset(
                ds,
                data_dict[ds],
                use_graph=True,
                use_temporal=True,
                use_pooling=True,
                use_cross_modal=False,
                enabled_modalities=("face", "eyes", "body"),
            )

            _ = run_cv_resume_safe(
                dataset_name=ds,
                model_name="MSTGNet",
                model_factory=mstg_factory,
                X_seq=X_seq,
                y_seq=y_seq,
                video_seq_ids=video_seq_ids,
                splits=splits_main[ds],
                seed=seed,
                save_checkpoint=True,
            )
            clean_memory()

            transformer_factory = build_transformer_for_dataset(
                ds,
                num_layers=int(CONFIG["transformer_num_layers"]),
            )

            _ = run_cv_resume_safe(
                dataset_name=ds,
                model_name=tr_name,
                model_factory=transformer_factory,
                X_seq=X_seq,
                y_seq=y_seq,
                video_seq_ids=video_seq_ids,
                splits=splits_main[ds],
                seed=seed,
                save_checkpoint=True,
            )
            clean_memory()

    multiseed_df = combine_fold_results([
        OUTPUT_ROOT / "fold_results" / "MSTGNet_I3D.csv",
        OUTPUT_ROOT / "fold_results" / "MSTGNet_RLT.csv",
        OUTPUT_ROOT / "fold_results" / f"{tr_name}_I3D.csv",
        OUTPUT_ROOT / "fold_results" / f"{tr_name}_RLT.csv",
    ])
    multiseed_df = drop_duplicate_fold_rows(multiseed_df)

    multiseed_df.to_csv(
        OUTPUT_ROOT / "stats" / "multiseed_results.csv",
        index=False,
    )

    stat_rows = []
    stat_json = {}

    for ds in DATASETS:
        wilcoxon_result = mb.wilcoxon_signed_rank_test(
            multiseed_df,
            model_a="MSTGNet",
            model_b=tr_name,
            metric="test_auc",
            dataset=ds,
        )

        ttest_result = mb.paired_ttest(
            multiseed_df,
            model_a="MSTGNet",
            model_b=tr_name,
            metric="test_auc",
            dataset=ds,
        )

        stat_json[ds] = {
            "wilcoxon_mstgnet_vs_transformer": wilcoxon_result,
            "paired_ttest_mstgnet_vs_transformer": ttest_result,
        }

        stat_rows.append(
            {
                "dataset": ds,
                "metric": "test_auc",
                "model_a": "MSTGNet",
                "model_b": tr_name,
                "wilcoxon_p_value": wilcoxon_result.get("p_value", np.nan),
                "wilcoxon_n_pairs": wilcoxon_result.get("n_pairs", np.nan),
                "paired_ttest_p_value": ttest_result.get("p_value", np.nan),
                "paired_ttest_n_pairs": ttest_result.get("n_pairs", np.nan),
                "mean_diff_mstgnet_minus_transformer": wilcoxon_result.get(
                    "mean_diff_a_minus_b",
                    np.nan,
                ),
            }
        )

    stats_df = pd.DataFrame(stat_rows)
    stats_df.to_csv(
        OUTPUT_ROOT / "stats" / "statistical_tests_summary.csv",
        index=False,
    )

    multiseed_summary = mb.summarize_cv_results(
        multiseed_df,
        metric_cols=[
            "test_auc",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_mcc",
            "test_f1",
        ],
        group_cols=["dataset", "model"],
    )

    save_table(multiseed_summary, "table_multiseed_robustness.csv")

    repeat_ci_metrics = [
        "test_auc",
        "test_f1",
        "test_mcc",
        "test_balanced_accuracy",
    ]

    repeat_ci_path = mb.save_repeat_level_ci_table(
        multiseed_df,
        OUTPUT_ROOT,
        filename="repeat_level_ci_multiseed.csv",
        metrics=repeat_ci_metrics,
        group_cols=["dataset", "model"],
        ci=0.95,
    )
    print(f"Saved repeat-level CI table: {repeat_ci_path}")

    for ds in DATASETS:
        ds_df = multiseed_df[multiseed_df["dataset"].astype(str) == str(ds)].copy()

        if len(ds_df) == 0:
            print(f"Skip repeat-level CI for {ds}: no data.")
            continue

        n_repeats_available = (
            int(ds_df["repeat"].nunique())
            if "repeat" in ds_df.columns else 1
        )
        print(f"\n{ds}: n_repeats_available={n_repeats_available}")

        ds_ci_parts = []

        for metric in repeat_ci_metrics:
            ci_df = mb.compute_repeat_level_ci(
                ds_df,
                metric=metric,
                group_cols=["dataset", "model"],
                ci=0.95,
            )

            if ci_df is not None and len(ci_df) > 0:
                ci_df["n_repeats_available"] = n_repeats_available
                ds_ci_parts.append(ci_df)

        ds_ci_all = (
            pd.concat(ds_ci_parts, ignore_index=True)
            if ds_ci_parts else pd.DataFrame()
        )

        ds_ci_path = OUTPUT_ROOT / "stats" / f"repeat_level_ci_{ds}.csv"
        ds_ci_all.to_csv(ds_ci_path, index=False)
        print(f"  Saved: {ds_ci_path} shape={ds_ci_all.shape}")

        preview_cols = [
            "dataset",
            "model",
            "metric",
            "repeat_n",
            "mean",
            "ci_low",
            "ci_high",
        ]
        existing_preview_cols = [c for c in preview_cols if c in ds_ci_all.columns]

        if len(ds_ci_all) > 0 and len(existing_preview_cols) > 0:
            print(ds_ci_all[existing_preview_cols].to_string(index=False))

    repeat_ci_all_parts = []

    for ds in DATASETS:
        p = OUTPUT_ROOT / "stats" / f"repeat_level_ci_{ds}.csv"
        if p.exists():
            repeat_ci_all_parts.append(pd.read_csv(p))

    repeat_ci_all = (
        pd.concat(repeat_ci_all_parts, ignore_index=True)
        if repeat_ci_all_parts else pd.DataFrame()
    )

    repeat_ci_all.to_csv(
        OUTPUT_ROOT / "stats" / "repeat_level_ci_all_datasets.csv",
        index=False,
    )

    save_table(repeat_ci_all, "table_repeat_level_ci.csv")
    print(f"\nRepeat-level CI all datasets: {repeat_ci_all.shape}")

    all_results_for_stats = combine_fold_results(baseline_fold_result_files())
    all_results_for_stats = drop_duplicate_fold_rows(all_results_for_stats)

    if len(all_results_for_stats) > 0:
        full_comparison_df = mb.compare_models_statistically(
            all_results_for_stats,
            reference_model="MSTGNet",
            metric="test_auc",
            adjust_pvalues=True,
        )

        full_comparison_df.to_csv(
            OUTPUT_ROOT / "stats" / "statistical_comparison_all_models.csv",
            index=False,
        )

        save_table(full_comparison_df, "table_statistical_comparison.csv")

        stat_json["all_model_comparison_note"] = (
            "Full comparison computed against all available paired fold results "
            "from baseline_fold_result_files(). Pairing uses dataset/seed/repeat/fold."
        )
    else:
        full_comparison_df = pd.DataFrame()
        warnings.warn(
            "No baseline fold results available for full statistical comparison.",
            RuntimeWarning,
        )

    mb.safe_json_dump(
        stat_json,
        OUTPUT_ROOT / "stats" / "statistical_tests.json",
    )

    print(stats_df)
    print(full_comparison_df)


# ============================================================
# S8 — Computational cost
# ============================================================

if RUN_S8_COMPUTE_COST:
    cost_sources = combine_fold_results([
        OUTPUT_ROOT / "fold_results" / "*.csv",
    ])
    cost_sources = drop_duplicate_fold_rows(cost_sources)

    cost_models = [
        "MSTGNet",
        "LSTM",
        "BiLSTM",
        "GRU",
        "CNN-LSTM",
        transformer_model_name(),
        "LogReg_temporal_stat",
        "RF_temporal_stat",
        "SVM_temporal_stat",
        "GradBoost_temporal_stat",
        "MSTGNet_NoGraph",
        "MSTGNet_NoTemporal",
        "MSTGNet_NoPooling",
        "MSTGNet_CrossModalAblation",
        "MSTGNet_FaceOnly",
        "MSTGNet_EyesOnly",
        "MSTGNet_BodyOnly",
        "MSTGNet_FaceEyes",
        "MSTGNet_FaceBody",
        "MSTGNet_EyesBody",
        "MSTGNet_Window100",
    ]

    cost_models += cross_dataset_model_names()

    if "model" in cost_sources.columns:
        cost_sources = cost_sources[cost_sources["model"].isin(cost_models)].copy()

    keep_cols = [
        "dataset",
        "source_dataset",
        "target_dataset",
        "model",
        "seed",
        "repeat",
        "fold",
        "n_params",
        "n_trainable_params",
        "n_total_params",
        "train_time_sec",
        "ms_per_batch",
        "ms_per_sequence",
        "ms_per_video",
        "n_train_seq",
        "n_val_seq",
        "n_test_seq",
        "n_train_videos",
        "n_val_videos",
        "n_test_videos",
        "n_source_train_videos",
        "n_source_val_videos",
        "n_target_adapt_videos",
        "n_target_test_videos",
    ]

    if len(cost_sources) > 0:
        existing_cols = [c for c in keep_cols if c in cost_sources.columns]
        cost_fold = cost_sources[existing_cols].copy()
    else:
        cost_fold = pd.DataFrame(columns=keep_cols)

    cost_fold.to_csv(
        OUTPUT_ROOT / "compute_cost" / "cost_table.csv",
        index=False,
    )

    param_col = get_param_column(cost_fold)

    if len(cost_fold) > 0:
        agg_kwargs = {
            "n_folds": ("fold", "count"),
            "train_time_sec_mean": ("train_time_sec", "mean"),
            "train_time_sec_std": ("train_time_sec", "std"),
            "ms_per_video_mean": ("ms_per_video", "mean"),
            "ms_per_video_std": ("ms_per_video", "std"),
            "ms_per_sequence_mean": ("ms_per_sequence", "mean"),
            "ms_per_sequence_std": ("ms_per_sequence", "std"),
        }

        if param_col is not None:
            agg_kwargs["n_params_mean"] = (param_col, "mean")
            agg_kwargs["n_params_median"] = (param_col, "median")

        cost_summary = (
            cost_fold
            .groupby(["dataset", "model"], dropna=False)
            .agg(**agg_kwargs)
            .reset_index()
        )
    else:
        cost_summary = pd.DataFrame()

    save_table(cost_summary, "table_computational_cost.csv")

    capacity_rows = []

    for ds in DATASETS:
        cdf = mb.compute_model_capacity_table(capacity_factories_for_dataset(ds))
        cdf["dataset"] = ds
        capacity_rows.append(cdf)

    capacity_df = (
        pd.concat(capacity_rows, ignore_index=True)
        if capacity_rows else pd.DataFrame()
    )

    capacity_df.to_csv(
        OUTPUT_ROOT / "compute_cost" / "model_capacity_table.csv",
        index=False,
    )

    print(capacity_df)


# ============================================================
# S9 — Calibration export
# ============================================================

if RUN_S9_CALIBRATION_EXPORT:
    pred_files = sorted((OUTPUT_ROOT / "predictions").glob("*video_predictions.csv"))

    pred_dfs = []

    for p in pred_files:
        try:
            d = pd.read_csv(p)
            d["source_file"] = str(p)
            pred_dfs.append(d)
        except Exception as e:
            warnings.warn(f"Could not read {p}: {e}", RuntimeWarning)

    pred_all = pd.concat(pred_dfs, ignore_index=True) if pred_dfs else pd.DataFrame()
    pred_all = drop_duplicate_prediction_rows(pred_all)

    pred_all.to_csv(
        OUTPUT_ROOT / "figures_data" / "all_video_predictions_for_calibration.csv",
        index=False,
    )

    if len(pred_all) > 0:
        cal_summary = (
            pred_all
            .groupby(["dataset", "model", "split"], dropna=False)
            .agg(
                n=("video_id", "count"),
                mean_y_true=("y_true", "mean"),
                mean_y_prob=("y_prob", "mean"),
                threshold_mean=("threshold", "mean"),
            )
            .reset_index()
        )
    else:
        cal_summary = pd.DataFrame()

    save_table(cal_summary, "table_calibration_prediction_summary.csv")
    print(cal_summary)


# ============================================================
# S10 — Interpretability
# ============================================================

if RUN_S10_INTERPRETABILITY:
    require_long_run("S10_INTERPRETABILITY")

    for ds in DATASETS:
        print("\n" + "#" * 90)
        print(f"S10 Interpretability — {ds}")
        print("#" * 90)

        ckpt_path = find_best_checkpoint(
            ds,
            "MSTGNet",
            seed_whitelist=[43, 44, 45, 46, 47],
            repeat_whitelist=[1],
        )

        if ckpt_path is None:
            print(f"{ds}: no checkpoint with seed_whitelist=[43..47], trying repeat=1 only.")
            ckpt_path = find_best_checkpoint(
                ds,
                "MSTGNet",
                repeat_whitelist=[1],
            )

        if ckpt_path is None:
            print(f"{ds}: no checkpoint with repeat=1 filter, trying no filter.")
            ckpt_path = find_best_checkpoint(
                ds,
                "MSTGNet",
            )

        if ckpt_path is None:
            warnings.warn(f"No checkpoint found for interpretability: {ds}", RuntimeWarning)
            continue

        factory = build_mstgnet_for_dataset(
            ds,
            data_dict[ds],
            use_graph=True,
            use_temporal=True,
            use_pooling=True,
            use_cross_modal=False,
            enabled_modalities=("face", "eyes", "body"),
        )

        X_seq, y_seq, video_seq_ids = get_sequences(ds)

        ckpt = safe_torch_load_checkpoint_local(ckpt_path, map_location="cpu")
        split = ckpt.get("split", ckpt.get("source_split", {}))
        split_videos = split.get("val_videos", None)

        imp_df = mb.compute_gradient_importance_from_checkpoint(
            checkpoint_path=ckpt_path,
            model_factory=factory,
            X_seq=X_seq,
            y_seq=y_seq,
            video_seq_ids=video_seq_ids,
            device=DEVICE,
            feature_names=data_dict[ds].get("coord_cols_active", None),
            modality_slices=data_dict[ds].get("active_modality_slices", None),
            split_videos=split_videos,
            batch_size=int(CONFIG["batch_size"]),
            max_sequences=2048,
            seed=SEED,
        )

        if imp_df is None or len(imp_df) == 0:
            warnings.warn(
                f"Empty importance dataframe for {ds}: {ckpt_path}",
                RuntimeWarning,
            )
            continue

        suffix = (
            f"seed{imp_df['seed'].iloc[0]}"
            f"_rep{imp_df['repeat'].iloc[0]}"
            f"_fold{imp_df['fold'].iloc[0]}"
        )

        paths = mb.save_gradient_importance(
            imp_df,
            OUTPUT_ROOT,
            dataset_name=ds,
            model_name="MSTGNet",
            top_k=100,
            suffix=suffix,
        )

        print("Saved gradient importance:", paths)

        imp_df.to_csv(
            OUTPUT_ROOT / "interpretability" / f"{ds}_MSTGNet_gradient_importance_all.csv",
            index=False,
        )

        clean_memory()


# ============================================================
# S11 — Export paper tables
# ============================================================

if RUN_S11_EXPORT_PAPER_TABLES:
    if not (OUTPUT_ROOT / "tables" / "table_baseline_comparison.csv").exists():
        baseline_all = combine_fold_results(baseline_fold_result_files())
        baseline_all = drop_duplicate_fold_rows(baseline_all)
        summarize_results(baseline_all, output_name="table_baseline_comparison.csv")

    for table_name in [
        "table_baseline_comparison.csv",
        "table_modality_ablation.csv",
        "table_architecture_ablation.csv",
        "table_multiseed_robustness.csv",
        "table_cross_dataset.csv",
        "table_computational_cost.csv",
        "table_temporal_sensitivity.csv",
        "table_statistical_comparison.csv",
        "table_landmark_quality_prediction_correlation.csv",
        "table_repeat_level_ci.csv",
    ]:
        standardize_table_for_paper_if_exists(table_name)

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "dataset_summary.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table1_dataset_summary.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_baseline_comparison.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table2_baseline_comparison.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_modality_ablation.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table3_modality_ablation.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_architecture_ablation.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table4_architecture_ablation.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_multiseed_robustness.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table5_multiseed_robustness.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_cross_dataset.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table6_cross_dataset.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_landmark_quality.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table7_landmark_quality.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_computational_cost.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table8_computational_cost.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_temporal_sensitivity.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table9_temporal_sensitivity.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_statistical_comparison.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table10_statistical_comparison.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_landmark_quality_prediction_correlation.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table11_landmark_quality_prediction_correlation.csv",
    )

    copy_if_exists(
        OUTPUT_ROOT / "tables" / "table_repeat_level_ci.csv",
        OUTPUT_ROOT / "tables" / "PAPER_Table12_repeat_level_ci.csv",
    )

    paper_tables = sorted((OUTPUT_ROOT / "tables").glob("PAPER_Table*.csv"))

    paper_index = pd.DataFrame(
        {
            "paper_table": [p.name for p in paper_tables],
            "path": [str(p) for p in paper_tables],
            "exists": [p.exists() for p in paper_tables],
        }
    )

    paper_index.to_csv(
        OUTPUT_ROOT / "tables" / "PAPER_Table_index.csv",
        index=False,
    )

    print(paper_index)


# ============================================================
# Final checklist
# ============================================================

final_files = {
    "manifest": OUTPUT_ROOT / "config" / "experiment_manifest.json",
    "dataset_summary": OUTPUT_ROOT / "tables" / "dataset_summary.csv",
    "capacity_check_pre_S2": OUTPUT_ROOT / "compute_cost" / "capacity_check_pre_S2.csv",
    "main_I3D": OUTPUT_ROOT / "fold_results" / "main_I3D.csv",
    "main_RLT": OUTPUT_ROOT / "fold_results" / "main_RLT.csv",
    "full_ref_I3D": OUTPUT_ROOT / "fold_results" / "full_model_reference_I3D.csv",
    "full_ref_RLT": OUTPUT_ROOT / "fold_results" / "full_model_reference_RLT.csv",
    "baseline_table": OUTPUT_ROOT / "tables" / "table_baseline_comparison.csv",
    "arch_table": OUTPUT_ROOT / "tables" / "table_architecture_ablation.csv",
    "modal_table": OUTPUT_ROOT / "tables" / "table_modality_ablation.csv",
    "landmark_table": OUTPUT_ROOT / "tables" / "table_landmark_quality.csv",
    "landmark_prediction_corr": OUTPUT_ROOT / "tables" / "table_landmark_quality_prediction_correlation.csv",
    "cross_dataset_table": OUTPUT_ROOT / "tables" / "table_cross_dataset.csv",
    "compute_cost_table": OUTPUT_ROOT / "tables" / "table_computational_cost.csv",
    "temporal_sensitivity_table": OUTPUT_ROOT / "tables" / "table_temporal_sensitivity.csv",
    "multiseed_stats": OUTPUT_ROOT / "stats" / "statistical_tests.json",
    "statistical_comparison": OUTPUT_ROOT / "stats" / "statistical_comparison_all_models.csv",
    "repeat_level_ci_I3D": OUTPUT_ROOT / "stats" / "repeat_level_ci_I3D.csv",
    "repeat_level_ci_RLT": OUTPUT_ROOT / "stats" / "repeat_level_ci_RLT.csv",
    "repeat_level_ci_all": OUTPUT_ROOT / "stats" / "repeat_level_ci_all_datasets.csv",
    "repeat_level_ci_multiseed": OUTPUT_ROOT / "tables" / "repeat_level_ci_multiseed.csv",
    "repeat_level_ci_table": OUTPUT_ROOT / "tables" / "table_repeat_level_ci.csv",
    "paper_table_index": OUTPUT_ROOT / "tables" / "PAPER_Table_index.csv",
    "paper_table12_repeat_ci": OUTPUT_ROOT / "tables" / "PAPER_Table12_repeat_level_ci.csv",
    "interpretability_dir": OUTPUT_ROOT / "interpretability",
}

final_check = pd.DataFrame(
    {
        "item": list(final_files.keys()),
        "path": [str(p) for p in final_files.values()],
        "exists": [Path(p).exists() for p in final_files.values()],
    }
)

final_check.to_csv(
    OUTPUT_ROOT / "tables" / "final_file_checklist.csv",
    index=False,
)

print(final_check)


Using device: cuda
OUTPUT_ROOT: outputs\revision_final_20260501
DEVICE: cuda
PASS: model_base API compatible with notebook
Loading processed dataset: data\processed\mstgnet_I3D.pkl
✓ I3D loaded
  Videos: 1568
  Observed features: 1533
  Active features:   1503
  Active slices:     {'face': [0, 1374], 'eyes': [1374, 1404], 'body': [1404, 1503]}
Loading processed dataset: data\processed\mstgnet_RLT.pkl
✓ RLT loaded
  Videos: 121
  Observed features: 1533
  Active features:   1503
  Active slices:     {'face': [0, 1374], 'eyes': [1374, 1404], 'body': [1404, 1503]}
Saved table: outputs\revision_final_20260501\tables\dataset_summary.csv shape=(2, 9)
  dataset  n_videos  n_class_0  n_class_1  n_observed_features  \
0     I3D      1568        784        784                 1533   
1     RLT       121         60         61                 1533   

   n_active_features face_slice    eyes_slice    body_slice  
0               1503  [0, 1374]  [1374, 1404]  [1404, 1503]  
1               1503  [0